In [2]:
#import statements
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import matplotlib.pyplot as plt
from torchinfo import summary
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
import nilearn.image
import nilearn.plotting
import copy
from torch.utils.data import random_split, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, r2_score
from sklearn.preprocessing import label_binarize
from pathlib import Path
from scipy import signal
import mne
from mne.preprocessing import ICA
from mne_icalabel.iclabel import iclabel_label_components

/Users/william.wakefield/PycharmProjects/Mayo_EEG_FDG_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
mne.set_log_level('WARNING')
INPUT_DIR = Path("model_data/orig_eeg_raw")
OUTPUT_DIR = Path("model_data/non_ica")

## Non-ICA pre-processing

In [ ]:
def process_subject_non_ica(df):
    start_ids = np.sort(df['start_index'].unique())
    first_block = df[df['start_index'] == start_ids[0]]
    channel_names = first_block['channel_name'].astype(str).tolist()
    dupes = [c for c in set(channel_names) if channel_names.count(c) > 1]
    if dupes:
        raise ValueError(
            f"Duplicate channel names {sorted(dupes)} within start_index "
            f"{start_ids[0]} — parquet contains overlapping recordings."
        )

    seg_arrays, seg_lengths = [], []
    for sid in start_ids:
        block = df[df['start_index'] == sid]
        block_chs = block['channel_name'].astype(str).tolist()
        if len(block_chs) != len(channel_names) or set(block_chs) != set(channel_names):
            raise ValueError(
                f"start_index {sid} has channels {block_chs}, "
                f"expected {channel_names}."
            )
        seg_data = np.stack([
            np.asarray(
                block.loc[block['channel_name'].astype(str) == ch, 'segment'].values[0],
                dtype=np.float64,
            )
            for ch in channel_names
        ])
        seg_arrays.append(seg_data)
        seg_lengths.append(seg_data.shape[1])
    data = np.concatenate(seg_arrays, axis=1)  # (n_channels, n_total_samples)

    info = mne.create_info(ch_names=channel_names, sfreq=256, ch_types='eeg')
    raw = mne.io.RawArray(data, info)
    raw.filter(l_freq=0.5, h_freq=45, method='fir',
               phase='zero', fir_window='hamming')
    raw.set_eeg_reference('average')

    arr = raw.get_data()
    ch_mean = arr.mean(axis=1, keepdims=True)
    ch_std = arr.std(axis=1, keepdims=True) + 1e-8
    arr = (arr - ch_mean) / ch_std

    out_blocks = []
    offset = 0
    for new_seg_idx, (sid, slen) in enumerate(zip(start_ids, seg_lengths)):
        seg_clean = arr[:, offset:offset + slen]
        offset += slen
        block = df[df['start_index'] == sid].copy()
        block['segment_index'] = new_seg_idx  # renumber to 0..N-1
        for ch_idx, ch in enumerate(channel_names):
            row_idx = block.index[block['channel_name'].astype(str) == ch][0]
            block.at[row_idx, 'segment'] = seg_clean[ch_idx].astype(np.float32)
        out_blocks.append(block)
    return pd.concat(out_blocks, ignore_index=True)

In [5]:
skipped = []
for parquet_path in sorted(INPUT_DIR.glob("*.parquet")):
    subject_id = parquet_path.stem
    print(f"Processing {subject_id} ... ", end="", flush=True)
    try:
        df = pd.read_parquet(parquet_path)
        df_out = process_subject_non_ica(df)
        df_out.to_parquet(OUTPUT_DIR / parquet_path.name, index=False)
        print("ok")
    except Exception as e:
        print(f"SKIPPED — {e}")
        skipped.append((subject_id, str(e)))

Processing 100584250 ... ok
Processing 100597363 ... ok
Processing 100864302 ... ok
Processing 101382732 ... ok
Processing 102841616 ... ok
Processing 102933094 ... ok
Processing 103293424 ... ok
Processing 104115732 ... ok
Processing 104388202 ... ok
Processing 104454780 ... ok
Processing 104794142 ... ok
Processing 104921540 ... ok
Processing 106169522 ... ok
Processing 106714847 ... ok
Processing 107333864 ... ok
Processing 107501813 ... ok
Processing 107943342 ... ok
Processing 107969830 ... ok
Processing 107992048 ... ok
Processing 108540365 ... ok
Processing 108853799 ... ok
Processing 109372236 ... ok
Processing 109400346 ... ok
Processing 109711792 ... ok
Processing 109801129 ... ok
Processing 109942570 ... ok
Processing 110034837 ... ok
Processing 110717532 ... ok
Processing 110770774 ... ok
Processing 110816623 ... ok
Processing 110871979 ... ok
Processing 111398473 ... ok
Processing 111452972 ... ok
Processing 111886762 ... ok
Processing 111991802 ... ok
Processing 112490764

In [6]:
pca_vals = pd.read_parquet("model_data/matched_pca_vectors.parquet")

### ICA

In [21]:
ICA_OUTPUT_DIR = Path("model_data/post_ica")

LABEL_NAMES = ['brain', 'muscle', 'eye', 'heart', 'line_noise', 'chan_noise', 'other']
LABEL_THRESHOLDS = {1: 0.90, 2: 0.50, 3: 0.50, 4: 0.70}

In [ ]:
def process_subject_ica(df):
    start_ids = np.sort(df['start_index'].unique())
    channel_names = df[df['start_index'] == start_ids[0]]['channel_name'].astype(str).tolist()

    seg_arrays, seg_lengths = [], []
    for sid in start_ids:
        block = df[df['start_index'] == sid]
        seg_data = np.stack([
            np.asarray(block.loc[block['channel_name'].astype(str) == ch,
                                 'segment'].values[0], dtype=np.float64)
            for ch in channel_names
        ])
        seg_arrays.append(seg_data)
        seg_lengths.append(seg_data.shape[1])
    data = np.concatenate(seg_arrays, axis=1)

    info = mne.create_info(ch_names=channel_names, sfreq=256, ch_types='eeg')
    raw  = mne.io.RawArray(data, info)
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='ignore')
    raw.set_eeg_reference('average', projection=False)
    raw.filter(l_freq=1.0, h_freq=100.0, method='fir', phase='zero', fir_window='hamming')
    raw_fit = raw.copy().crop(tmax=min(raw.times[-1], 120.0))

    n_comp = 7
    ica = ICA(n_components=n_comp, method='infomax',
              fit_params=dict(extended=True), random_state=42, max_iter=800)
    ica.fit(raw_fit)

    excluded = []
    try:
        label_probs = iclabel_label_components(raw_fit, ica)  # (n_comp, 7)
        for i, probs in enumerate(label_probs):
            pred = int(np.argmax(probs))
            thresh = LABEL_THRESHOLDS.get(pred)
            if thresh is not None and probs[pred] >= thresh:
                ica.exclude.append(i)
                excluded.append({
                    'comp': i,
                    'label': LABEL_NAMES[pred],
                    'p': float(probs[pred]),
                    'all_probs': probs.tolist(),
                })
    except Exception as e:
        label_probs = None
        print(f"  [ICLabel failed: {e}]")

    ica.apply(raw)
    raw.filter(l_freq=None, h_freq=45.0, method='fir', phase='zero', fir_window='hamming')
    arr = raw.get_data()
    arr = (arr - arr.mean(axis=1, keepdims=True)) / (arr.std(axis=1, keepdims=True) + 1e-8)

    out_blocks, offset = [], 0
    for new_idx, (sid, slen) in enumerate(zip(start_ids, seg_lengths)):
        seg_clean = arr[:, offset:offset + slen]
        offset += slen
        block = df[df['start_index'] == sid].copy()
        block['segment_index'] = new_idx
        for ch_idx, ch in enumerate(channel_names):
            row = block.index[block['channel_name'].astype(str) == ch][0]
            block.at[row, 'segment'] = seg_clean[ch_idx].astype(np.float32)
        out_blocks.append(block)

    return pd.concat(out_blocks, ignore_index=True), label_probs, excluded

In [28]:
skipped_ica = []
hdr = f"  {'IC':>3}  " + "  ".join(f"{n:>10}" for n in LABEL_NAMES)
paths = sorted(INPUT_DIR.glob("*.parquet"))
n = len(paths)

In [30]:
for idx, parquet_path in enumerate(paths, 1):
    subject_id = parquet_path.stem
    print(f"\n[{idx}/{n}] {subject_id}")
    try:
        df = pd.read_parquet(parquet_path)
        df_out, label_probs, excluded = process_subject_ica(df)
        df_out.to_parquet(ICA_OUTPUT_DIR / parquet_path.name, index=False)

        if label_probs is not None:
            print(hdr)
            for i, probs in enumerate(label_probs):
                marker = " ◄" if i in {e['comp'] for e in excluded} else ""
                print(f"  {i:>3}  " + "  ".join(f"{p:>10.3f}" for p in probs) + marker)

        if excluded:
            summary = ", ".join(f"IC{e['comp']}={e['label']}({e['p']:.2f})" for e in excluded)
            print(f"  → excluded {len(excluded)}: {summary}")
        else:
            print("  → excluded 0 components")

    except Exception as e:
        import traceback
        print(f"  SKIPPED — {e}")
        traceback.print_exc()
        skipped_ica.append((subject_id, str(e)))


[1/1582] 100584250
   IC       brain      muscle         eye       heart  line_noise  chan_noise       other
    0       0.739       0.003       0.005       0.022       0.002       0.002       0.227
    1       0.920       0.017       0.008       0.001       0.009       0.003       0.043
    2       0.063       0.069       0.402       0.095       0.006       0.000       0.365
    3       0.965       0.001       0.000       0.001       0.008       0.000       0.025
    4       0.921       0.002       0.000       0.002       0.005       0.009       0.061
    5       0.990       0.000       0.000       0.000       0.001       0.000       0.008
    6       0.108       0.003       0.001       0.358       0.161       0.003       0.365
  → excluded 0 components

[2/1582] 100597363
   IC       brain      muscle         eye       heart  line_noise  chan_noise       other
    0       0.316       0.002       0.005       0.009       0.015       0.064       0.590
    1       0.923       0.015     

/var/folders/4r/2w_vg8k91mldqsvxyw1y1cnml0bjbd/T/ipykernel_18289/3526321573.py:31: RuntimeWarning: Using n_components=7 (resulting in n_components_=7) may lead to an unstable mixing matrix estimation because the ratio between the largest (8) and smallest (3.6e-06) variances is too large (> 1e6); consider setting n_components=0.999999 or an integer <= 6
  ica.fit(raw_fit)


   IC       brain      muscle         eye       heart  line_noise  chan_noise       other
    0       0.858       0.001       0.002       0.003       0.006       0.018       0.112
    1       0.314       0.001       0.523       0.029       0.002       0.007       0.124 ◄
    2       0.930       0.001       0.001       0.005       0.003       0.016       0.043
    3       0.773       0.019       0.003       0.029       0.009       0.009       0.157
    4       0.409       0.001       0.006       0.004       0.011       0.007       0.562
    5       0.405       0.003       0.060       0.074       0.004       0.035       0.417
    6       0.813       0.001       0.029       0.014       0.019       0.008       0.115
  → excluded 1: IC1=eye(0.52)

[166/1582] 179859039
   IC       brain      muscle         eye       heart  line_noise  chan_noise       other
    0       0.367       0.141       0.205       0.075       0.018       0.028       0.166
    1       0.913       0.000       0.002     

/var/folders/4r/2w_vg8k91mldqsvxyw1y1cnml0bjbd/T/ipykernel_18289/3526321573.py:31: RuntimeWarning: Using n_components=7 (resulting in n_components_=7) may lead to an unstable mixing matrix estimation because the ratio between the largest (8) and smallest (5.6e-06) variances is too large (> 1e6); consider setting n_components=0.999999 or an integer <= 6
  ica.fit(raw_fit)


   IC       brain      muscle         eye       heart  line_noise  chan_noise       other
    0       0.294       0.002       0.003       0.010       0.028       0.076       0.588
    1       0.985       0.000       0.000       0.000       0.000       0.000       0.014
    2       0.993       0.000       0.000       0.000       0.001       0.000       0.005
    3       0.968       0.000       0.000       0.000       0.001       0.000       0.030
    4       0.994       0.000       0.000       0.001       0.001       0.000       0.004
    5       0.112       0.005       0.487       0.047       0.025       0.012       0.312
    6       0.754       0.002       0.002       0.004       0.034       0.002       0.202
  → excluded 0 components

[332/1582] 274717441
   IC       brain      muscle         eye       heart  line_noise  chan_noise       other
    0       0.587       0.003       0.072       0.040       0.005       0.005       0.289
    1       0.989       0.000       0.000       0.00

/var/folders/4r/2w_vg8k91mldqsvxyw1y1cnml0bjbd/T/ipykernel_18289/3526321573.py:31: RuntimeWarning: Using n_components=7 (resulting in n_components_=7) may lead to an unstable mixing matrix estimation because the ratio between the largest (8) and smallest (7.8e-06) variances is too large (> 1e6); consider setting n_components=0.999999 or an integer <= 6
  ica.fit(raw_fit)


   IC       brain      muscle         eye       heart  line_noise  chan_noise       other
    0       0.641       0.002       0.023       0.009       0.001       0.018       0.306
    1       0.316       0.040       0.162       0.058       0.002       0.008       0.414
    2       0.630       0.075       0.031       0.059       0.005       0.031       0.169
    3       0.673       0.001       0.002       0.018       0.010       0.003       0.293
    4       0.895       0.022       0.001       0.008       0.005       0.005       0.063
    5       0.903       0.001       0.004       0.006       0.006       0.002       0.078
    6       0.841       0.001       0.001       0.034       0.091       0.001       0.031
  → excluded 0 components

[566/1582] 422530439
   IC       brain      muscle         eye       heart  line_noise  chan_noise       other
    0       0.624       0.001       0.030       0.019       0.002       0.002       0.324
    1       0.849       0.002       0.000       0.00

# Non-ICA processing for 19 channel data.

In [9]:
mne.set_log_level('WARNING')
INPUT_DIR = Path("model_data/orig_eeg_raw_19_channels")
OUTPUT_DIR = Path("model_data/non_ica_19_channels")

In [11]:
def process_subject_non_ica(df):
    start_ids = np.sort(df['start_index'].unique())
    first_block = df[df['start_index'] == start_ids[0]]
    channel_names = first_block['channel_name'].astype(str).tolist()
    dupes = [c for c in set(channel_names) if channel_names.count(c) > 1]
    if dupes:
        raise ValueError(
            f"Duplicate channel names {sorted(dupes)} within start_index "
            f"{start_ids[0]} — parquet contains overlapping recordings."
        )

    seg_arrays, seg_lengths = [], []
    for sid in start_ids:
        block = df[df['start_index'] == sid]
        block_chs = block['channel_name'].astype(str).tolist()
        if len(block_chs) != len(channel_names) or set(block_chs) != set(channel_names):
            raise ValueError(
                f"start_index {sid} has channels {block_chs}, "
                f"expected {channel_names}."
            )
        seg_data = np.stack([
            np.asarray(
                block.loc[block['channel_name'].astype(str) == ch, 'segment'].values[0],
                dtype=np.float64,
            )
            for ch in channel_names
        ])
        seg_arrays.append(seg_data)
        seg_lengths.append(seg_data.shape[1])
    data = np.concatenate(seg_arrays, axis=1)  # (n_channels, n_total_samples)

    info = mne.create_info(ch_names=channel_names, sfreq=256, ch_types='eeg')
    raw = mne.io.RawArray(data, info)
    raw.filter(l_freq=0.5, h_freq=45, method='fir',
               phase='zero', fir_window='hamming')
    raw.set_eeg_reference('average')

    arr = raw.get_data()
    ch_mean = arr.mean(axis=1, keepdims=True)
    ch_std = arr.std(axis=1, keepdims=True) + 1e-8
    arr = (arr - ch_mean) / ch_std

    out_blocks = []
    offset = 0
    for new_seg_idx, (sid, slen) in enumerate(zip(start_ids, seg_lengths)):
        seg_clean = arr[:, offset:offset + slen]
        offset += slen
        block = df[df['start_index'] == sid].copy()
        block['segment_index'] = new_seg_idx  # renumber to 0..N-1
        for ch_idx, ch in enumerate(channel_names):
            row_idx = block.index[block['channel_name'].astype(str) == ch][0]
            block.at[row_idx, 'segment'] = seg_clean[ch_idx].astype(np.float32)
        out_blocks.append(block)
    return pd.concat(out_blocks, ignore_index=True)

In [9]:
skipped = []
for parquet_path in sorted(INPUT_DIR.glob("*.parquet")):
    subject_id = parquet_path.stem
    print(f"Processing {subject_id} ... ", end="", flush=True)
    try:
        df = pd.read_parquet(parquet_path)
        df_out = process_subject_non_ica(df)
        df_out.to_parquet(OUTPUT_DIR / parquet_path.name, index=False)
        print("ok")
    except Exception as e:
        print(f"SKIPPED — {e}")
        skipped.append((subject_id, str(e)))

Processing 100584250 ... ok
Processing 100597363 ... ok
Processing 100864302 ... ok
Processing 101382732 ... ok
Processing 102841616 ... ok
Processing 102933094 ... ok
Processing 103293424 ... ok
Processing 104115732 ... ok
Processing 104388202 ... ok
Processing 104454780 ... ok
Processing 104794142 ... ok
Processing 104921540 ... ok
Processing 106169522 ... ok
Processing 106714847 ... ok
Processing 107333864 ... ok
Processing 107501813 ... ok
Processing 107943342 ... ok
Processing 107969830 ... ok
Processing 107992048 ... ok
Processing 108540365 ... ok
Processing 108853799 ... ok
Processing 109372236 ... ok
Processing 109400346 ... ok
Processing 109711792 ... ok
Processing 109801129 ... ok
Processing 109942570 ... ok
Processing 110034837 ... ok
Processing 110717532 ... ok
Processing 110770774 ... ok
Processing 110816623 ... ok
Processing 110871979 ... ok
Processing 111398473 ... ok
Processing 111452972 ... ok
Processing 111886762 ... ok
Processing 111991802 ... ok
Processing 112490764

# ICA processing for 19 channel data

In [10]:
ICA_OUTPUT_DIR = Path("model_data/post_ica_19_channels_var_2")

LABEL_NAMES = ['brain', 'muscle', 'eye', 'heart', 'line_noise', 'chan_noise', 'other']
ARTIFACT_THRESHOLD = 0.9

In [11]:
def process_subject_ica(df):
    start_ids = np.sort(df['start_index'].unique())
    channel_names = df[df['start_index'] == start_ids[0]]['channel_name'].astype(str).tolist()

    seg_arrays, seg_lengths = [], []
    for sid in start_ids:
        block = df[df['start_index'] == sid]
        seg_data = np.stack([
            np.asarray(block.loc[block['channel_name'].astype(str) == ch,
                                 'segment'].values[0], dtype=np.float64)
            for ch in channel_names
        ])
        seg_arrays.append(seg_data)
        seg_lengths.append(seg_data.shape[1])
    data = np.concatenate(seg_arrays, axis=1)

    info = mne.create_info(ch_names=channel_names, sfreq=256, ch_types='eeg')
    raw  = mne.io.RawArray(data, info)
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='ignore')
    raw.set_eeg_reference('average', projection=False)
    raw.filter(l_freq=1.0, h_freq=100.0, method='fir', phase='zero', fir_window='hamming')
    raw_fit = raw.copy().crop(tmax=min(raw.times[-1], 120.0))

    try:
        ica = ICA(n_components=0.999, method='infomax',
                  fit_params=dict(extended=True), random_state=42, max_iter=800)
        ica.fit(raw_fit)
    except RuntimeError:
        # One component dominates variance; fall back to fixed n_components=2
        print("  [variance fallback: n_components=2]")
        ica = ICA(n_components=2, method='infomax',
                  fit_params=dict(extended=True), random_state=42, max_iter=800)
        ica.fit(raw_fit)

    excluded = []
    try:
        label_probs = iclabel_label_components(raw_fit, ica)  # (n_comp, 7)
        for i, probs in enumerate(label_probs):
            # exclude if muscle, eye, or heart exceeds threshold
            if probs[1] > ARTIFACT_THRESHOLD or probs[2] > ARTIFACT_THRESHOLD or probs[3] > ARTIFACT_THRESHOLD:
                ica.exclude.append(i)
                pred = int(np.argmax(probs))
                excluded.append({
                    'comp': i,
                    'label': LABEL_NAMES[pred],
                    'p': float(probs[pred]),
                    'all_probs': probs.tolist(),
                })
    except Exception as e:
        label_probs = None
        print(f"  [ICLabel failed: {e}]")

    ica.apply(raw)
    raw.filter(l_freq=None, h_freq=45.0, method='fir', phase='zero', fir_window='hamming')
    arr = raw.get_data()
    arr = (arr - arr.mean(axis=1, keepdims=True)) / (arr.std(axis=1, keepdims=True) + 1e-8)

    out_blocks, offset = [], 0
    for new_idx, (sid, slen) in enumerate(zip(start_ids, seg_lengths)):
        seg_clean = arr[:, offset:offset + slen]
        offset += slen
        block = df[df['start_index'] == sid].copy()
        block['segment_index'] = new_idx
        for ch_idx, ch in enumerate(channel_names):
            row = block.index[block['channel_name'].astype(str) == ch][0]
            block.at[row, 'segment'] = seg_clean[ch_idx].astype(np.float32)
        out_blocks.append(block)

    return pd.concat(out_blocks, ignore_index=True), label_probs, excluded

In [12]:
skipped_ica = []
hdr = f"  {'IC':>3}  " + "  ".join(f"{n:>10}" for n in LABEL_NAMES)
paths = sorted(INPUT_DIR.glob("*.parquet"))
n = len(paths)

In [ ]:
for idx, parquet_path in enumerate(paths, 1):
    subject_id = parquet_path.stem
    out_path = ICA_OUTPUT_DIR / parquet_path.name
    if out_path.exists():
        continue

    print(f"\n[{idx}/{n}] {subject_id}")
    try:
        df = pd.read_parquet(parquet_path)
        df_out, label_probs, excluded = process_subject_ica(df)
        df_out.to_parquet(out_path, index=False)

        if label_probs is not None:
            print(hdr)
            for i, probs in enumerate(label_probs):
                marker = " ◄" if i in {e['comp'] for e in excluded} else ""
                print(f"  {i:>3}  " + "  ".join(f"{p:>10.3f}" for p in probs) + marker)

        if excluded:
            summary = ", ".join(f"IC{e['comp']}={e['label']}({e['p']:.2f})" for e in excluded)
            print(f"  → excluded {len(excluded)}: {summary}")
        else:
            print("  → excluded 0 components")

    except Exception as e:
        import traceback
        print(f"  SKIPPED — {e}")
        traceback.print_exc()
        skipped_ica.append((subject_id, str(e)))


[1/1574] 100584250
   IC       brain      muscle         eye       heart  line_noise  chan_noise       other
    0       0.007       0.019       0.872       0.006       0.002       0.058       0.036
    1       0.892       0.045       0.003       0.002       0.003       0.001       0.053
    2       0.000       0.000       0.996       0.000       0.000       0.000       0.003 ◄
    3       0.004       0.269       0.702       0.001       0.001       0.006       0.016
    4       0.100       0.578       0.021       0.008       0.005       0.005       0.282
    5       0.484       0.027       0.001       0.006       0.005       0.007       0.469
    6       0.943       0.010       0.000       0.009       0.010       0.001       0.027
    7       0.283       0.017       0.004       0.002       0.011       0.006       0.677
    8       0.969       0.000       0.000       0.000       0.004       0.000       0.025
    9       0.330       0.513       0.000       0.032       0.000       0.009 